In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Sun Mar 29 20:49:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 140.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 39.6 MB/s eta 0:

# Configuration

In [4]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [11]:
# Model configuration
# MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'
MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'

# Data configuration
SFT_DATA_ID = 'jxie/coco_captions'
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'
DATA_SPLIT = 'train'
SAVE_DATA_ID = 'alxxtexxr/BIVR-Data'

# For small experiment
DATA_SIZE = 100
BATCH_SIZE = 10

# # For larger experiment
# DATA_SIZE = 1000
# BATCH_SIZE = 100

# Model

In [ ]:
from unsloth import FastVisionModel
import torch
from transformers import AutoProcessor

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

# Data

In [12]:
from datasets import load_dataset, Dataset
from collections import deque

def load_hf_dataset(data_id, split, total_size, from_end=False):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)
    unique_cocoids = set()

    if not from_end:
        # Slice from the start (original behavior)
        sliced_data = []
        for example in dataset_stream:
            if len(sliced_data) >= total_size:
                break

            cocoid = example.get('cocoid', None)
            if cocoid is not None:
                if cocoid in unique_cocoids:
                    continue
                unique_cocoids.add(cocoid)

            sliced_data.append(example)
    else:
        # Slice from the end using a fixed-size buffer
        buffer = deque(maxlen=total_size)

        for example in dataset_stream:
            cocoid = example.get('cocoid', None)
            if cocoid is not None:
                if cocoid in unique_cocoids:
                    continue
                unique_cocoids.add(cocoid)

            buffer.append(example)

        sliced_data = list(buffer)

    dataset = Dataset.from_list(sliced_data)
    return dataset

sft_dataset = load_hf_dataset(SFT_DATA_ID, split=DATA_SPLIT, total_size=DATA_SIZE, from_end=False)
rl_dataset = load_hf_dataset(RL_DATA_ID, split=DATA_SPLIT, total_size=DATA_SIZE, from_end=False)

print("SFT dataset:")
print(sft_dataset)
print()
print("RL dataset:")
print(rl_dataset)

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

SFT dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 100
})

RL dataset:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 100
})


In [13]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

# if 'url' in dataset.features:
#     # Load and process images from URLs
#     dataset = dataset.map(
#         process_image_with_url, 
#         num_proc=4,
#     )
# else:
#     # Resize to (512, 512) and convert to RGB
#     dataset = dataset.map(
#         process_image, 
#         # num_proc=4, # Somehow multiprocessing causes errors on T4
#     )

sft_dataset = sft_dataset.map(
    process_image,
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)
rl_dataset = rl_dataset.map(
    process_image, 
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
from datasets import concatenate_datasets

# Keep only decoded_image + add source label
sft_images = sft_dataset.select_columns(['decoded_image']).map(lambda x: {'source': 'sft'})
rl_images = rl_dataset.select_columns(['decoded_image']).map(lambda x: {'source': 'rl'})

# Combine
train_dataset = concatenate_datasets([sft_images, rl_images])

# Rename column
train_dataset = train_dataset.rename_column('decoded_image', 'image')

print(train_dataset)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'source'],
    num_rows: 200
})


In [23]:
model.visual

Qwen3VLVisionModel(
  (patch_embed): Qwen3VLVisionPatchEmbed(
    (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (pos_embed): Embedding(2304, 1152)
  (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
  (blocks): ModuleList(
    (0-26): 27 x Qwen3VLVisionBlock(
      (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
      (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
      (attn): Qwen3VLVisionAttention(
        (qkv): Linear(in_features=1152, out_features=3456, bias=True)
        (proj): Linear(in_features=1152, out_features=1152, bias=True)
      )
      (mlp): Qwen3VLVisionMLP(
        (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
        (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
        (act_fn): GELUTanh()
      )
    )
  )
  (merger): Qwen3VLVisionPatchMerger(
    (norm): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
    (linear_fc1): Linear(in_features=4608, out_f

In [27]:
import math
import os
from pathlib import Path
import numpy as np
from huggingface_hub import upload_file
from sklearn.preprocessing import StandardScaler

train_size = len(train_dataset)
save_dir = Path(
    f"model_{MODEL_ID.replace('/', '_')}"
    f".data_sft_{SFT_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
    f".data_rl_{RL_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
)
os.makedirs(save_dir, exist_ok=True)

scaler = StandardScaler()

for i in range(math.ceil(train_size / BATCH_SIZE)):
    start_idx = i * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)
        
    # Take the first element as visual embeddings if it's a tuple (for compatibility with different model versions)
    if isinstance(visual_embs, tuple):
        visual_embs = visual_embs[0]

    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    # batch = visual_embs.float() # Convert to float32 for statistics calculation -> (N, 2048)

    # # Statistics for Standardization
    # batch_n = batch.shape[0]
    # batch_sum = batch.sum(dim=0)
    # batch_sum_sq = (batch**2).sum(dim=0)

    # # Statistics for Min-Max Normalization
    # batch_min = batch.min(dim=0).values
    # batch_max = batch.max(dim=0).values

    # print("Batch size:", batch_n)
    # print("Batch sum average:", batch_sum.mean())
    # print("Batch sum squared average:", batch_sum_sq.mean())
    # print("Batch min average:", batch_min.mean())
    # print("Batch max average:", batch_max.mean())
    # print()

    # Partial fit the scaler with current batch
    scaler.partial_fit(visual_embs.detach().cpu().float().numpy())

    # Save vision data and statistics
    range_tag = f'{str(start_idx).zfill(len(str(DATA_SIZE)))}-{str(end_idx-1).zfill(len(str(DATA_SIZE)))}'
    vision_data_path = save_dir / f'vision_data_{range_tag}.npz'
    # stats_path = save_dir / f'{range_tag}.stats.npz'
    
    np.savez(
        vision_data_path,
        pixel_values=pixel_values.cpu().numpy(),
        image_grid_thw=image_grid_thw.cpu().numpy(),
        visual_embs=visual_embs.detach().cpu().float().numpy(),
    )
    # np.savez(
    #     stats_path,
    #     batch_n=batch_n,
    #     batch_sum=batch_sum.cpu().numpy(),
    #     batch_sum_sq=batch_sum_sq.cpu().numpy(),
    #     batch_min=batch_min.cpu().numpy(),
    #     batch_max=batch_max.cpu().numpy()
    # )

    print("Vision data saved to:", vision_data_path)
    # print("Statistics saved to:", stats_path)

    upload_file(
        path_or_fileobj=str(vision_data_path),
        path_in_repo=str(vision_data_path),
        repo_id=SAVE_DATA_ID,
        repo_type='dataset',
    )
    # upload_file(
    #     path_or_fileobj=str(stats_path),
    #     path_in_repo=str(stats_path),
    #     repo_id=SAVE_DATA_ID,
    #     repo_type='dataset',
    # )

Range: 0-9
Pixel values shape: torch.Size([10240, 1536])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([2560, 4096])

Vision data saved to: model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100/vision_data_000-009.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0/vision_data_000-009.npz:   2%|1         | 1.86MB /  105MB            

Range: 10-19
Pixel values shape: torch.Size([10240, 1536])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([2560, 4096])

Vision data saved to: model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100/vision_data_010-019.npz


...

  ...0/vision_data_180-189.npz:   1%|1         | 1.44MB /  105MB            

Range: 190-199
Pixel values shape: torch.Size([10240, 1536])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([2560, 4096])

Vision data saved to: model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100/vision_data_190-199.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0/vision_data_190-199.npz:   8%|7         | 8.39MB /  105MB            

In [28]:
import joblib

# Save the scaler
scaler_path = save_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

upload_file(
    path_or_fileobj=str(scaler_path),
    path_in_repo=str(scaler_path),
    repo_id='alxxtexxr/BIVR-Data',
    repo_type='dataset',
)

Scaler saved to: model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100/scaler.pkl


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._ViRL1.25K_100/scaler.pkl: 100%|##########| 98.9kB / 98.9kB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/BVIR-Data/commit/241a98da3852bd42b9e207ed7898bd166c52025b', commit_message='Upload model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100/scaler.pkl with huggingface_hub', commit_description='', oid='241a98da3852bd42b9e207ed7898bd166c52025b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/BVIR-Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/BVIR-Data'), pr_revision=None, pr_num=None)